# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

**Выполнение:** запустите ячейки сверху вниз. Для повторного запуска в Google Colab загрузите `orders.csv` из LMS при появлении диалога. CSV в репозиторий сдавать не нужно.


## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения (до запуска проверочной ячейки):

1. `[1. 2. 3.] [0 1 1]` - обычное `/` выполняет деление с дробным результатом, `//` — целочисленное деление с округлением вниз.
2. `[4. 4. 4.]` — `m` имеет форму `(3, 4)` и состоит из единиц; `axis=1` складывает четыре числа отдельно в каждой из трех строк.
3. Матрица 3 × 3: `[[0, 1, 2], [1, 2, 3], [2, 3, 4]]` - broadcasting складывает столбец формы `(3, 1)` со строкой формы `(3,)`.
4. `6.0` - корни из `[1, 4, 9]` равны `[1., 2., 3.]`, их сумма равна 6.
5. `7.5` - квадраты чисел `[1, 2, 3, 4]` равны `[1, 4, 9, 16]`, среднее равно 30/4.


In [10]:
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())

[1. 2. 3.] [0 1 1]
[4. 4. 4.]
[[0 1 2]
 [1 2 3]
 [2 3 4]]
6.0
7.5


In [11]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [12]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [13]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [19]:
months = np.array(["2026-09", "2026-10"])
monthly_revenue = np.array([amount[clean & (month == m)].sum() for m in months])
change_pct = (monthly_revenue[1] / monthly_revenue[0] - 1) * 100

print(f"Сентябрь: {monthly_revenue[0]:,.0f}")
print(f"Октябрь: {monthly_revenue[1]:,.0f}")
print(f"Динамика изменения: {change_pct:+.2f}%")
print("Совпадает с выводом дз 2")

Сентябрь: 35,243,410
Октябрь: 34,419,520
Динамика изменения: -2.34%
Совпадает с выводом дз 2


## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [16]:
categories = np.unique(cat)
category_table = np.array([
    [amount[clean & (cat == category) & (month == m)].sum() for m in months]
    for category in categories
])
category_totals = category_table.sum(axis=1)
category_change_pct = (category_table[:, 1] / category_table[:, 0] - 1) * 100
leader_idx = category_totals.argmax()
fall_idx = category_change_pct.argmin()

print("Категория            Сентябрь      Октябрь      Всего      Изм., %")
for c, revenue, total, pct in zip(categories, category_table, category_totals, category_change_pct):
    print(f"{c:<18} {revenue[0]:>11,.0f} {revenue[1]:>12,.0f} {total:>12,.0f} {pct:>+9.2f}%")
print(f"Больще всего выручки за 2 мес: {categories[leader_idx]} - {category_totals[leader_idx]:,.0f}")
print(f"Сильнее всех упала в %: {categories[fall_idx]} - {category_change_pct[fall_idx]:+.2f}")

Категория            Сентябрь      Октябрь      Всего      Изм., %
Дом                  4,997,730    5,125,790   10,123,520     +2.56%
Книги                1,065,300    1,140,910    2,206,210     +7.10%
Красота              2,610,950    2,678,990    5,289,940     +2.61%
Одежда               9,167,100    8,505,730   17,672,830     -7.21%
Электроника         17,402,330   16,968,100   34,370,430     -2.50%
Больще всего выручки за 2 мес: Электроника - 34,370,430
Сильнее всех упала в %: Одежда - -7.21


## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [18]:
category_counts = np.array([
    [(clean & (cat == category) & (month == m)).sum() for m in months]
    for category in categories
])
category_avg_check = np.divide(category_table, category_counts, out=np.zeros_like(category_table), where=category_counts != 0)
october_avg = category_avg_check[:, 1]
best_oct_idx = october_avg.argmax()

print("Категория            Заказов сен/окт     Средний чек сен/окт")
for c, counts, avgs in zip(categories, category_counts, category_avg_check):
    print(f"{c:<18} {counts[0]:>5} / {counts[1]:<5} {avgs[0]:>13,.2f} / {avgs[1]:>11,.2f}")
print("Самый высокий средний чек - октябрь, электроника: 33,600.20")

Категория            Заказов сен/окт     Средний чек сен/окт
Дом                  476 / 476       10,499.43 /   10,768.47
Книги                265 / 299        4,020.00 /    3,815.75
Красота              422 / 435        6,187.09 /    6,158.60
Одежда               678 / 652       13,520.80 /   13,045.60
Электроника          531 / 505       32,772.75 /   33,600.20
Самый высокий средний чек - октябрь, электроника: 33,600.20


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [20]:
# 4

## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [21]:
# 5

## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [22]:
# 6

## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [23]:
# 7

## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [24]:
# 8